# Spilled Energy — T4 run

Runs **all estimators (baselines + SpilledEnergy) in a single UEManager pass**
on the frozen config, and reports **normalized PRR@0.5**.

**Runtime → Change runtime type → T4 GPU** first. All logic lives in
`harness/*.py` in the repo; these cells only orchestrate, so nothing here can
drift from the code in the PR.

## 1. Confirm you actually got a T4

Colab silently hands out different accelerators. Check before spending an hour.

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU'
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info()
print(f'device      : {name}')
print(f'VRAM free   : {free/1e9:.2f} GB / {total/1e9:.2f} GB')
if 'T4' not in name:
    print(f'\nWARNING: expected a T4, got {name!r}. Timings and the fp16-only\n'
          'constraint were chosen for a T4; results are still valid but speed differs.')

## 2. Mount Drive

Results are written **straight to Drive**, not copied at the end: the manager is
saved inside lm-polygraph's `finally:` block, so it survives even if the run
raises. Losing a long run to a disconnect is the most likely way this goes wrong.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/spilled_energy/runs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('results ->', DRIVE_OUT)

## 3. Clone the experiments branch

`spilled-energy-experiments` = the PR branch **plus** `harness/` and this notebook.
The PR itself is opened from `spilled-energy`, which contains only the
StatCalculator, the Estimator, the tests and the config.

In [ ]:
REPO_URL = 'https://github.com/<your-user>/lm-polygraph'  # <-- EDIT: your fork
BRANCH   = 'spilled-energy-experiments'

import os
if not os.path.isdir('/content/lm-polygraph'):
    !git clone --branch $BRANCH $REPO_URL /content/lm-polygraph
%cd /content/lm-polygraph
!git log -1 --oneline

## 4. Install

Colab already ships a CUDA torch; we install lm-polygraph and pin the rest.
Restart the runtime **only** if pip reports a version conflict it had to resolve.

In [ ]:
!pip install -q -e .
!pip install -q -r harness/requirements-repro.txt

import sysconfig, os
os.environ['PATH'] = os.environ['PATH'] + ':' + sysconfig.get_path('scripts')
!which polygraph_eval || echo 'NOTE: not on PATH; the runner prints a fallback'

## 5. Model provenance — what this config ACTUALLY resolves to

Printed before anything else touches the model. `--expect` makes a silent model
swap fail the run rather than produce numbers of unclear origin. Read the
`model.path`, `dtype` and `device` lines and confirm them against the report.

In [ ]:
!python harness/provenance.py --config configs/stage1/eval_triviaqa_qwen.yaml --expect Qwen/Qwen2.5-3B-Instruct

## 6. Run the unit tests (seconds, CPU)

Cheap guard that the install is sane before the long run.

In [ ]:
!python -m pytest test/test_spilled_energy.py -q

## 7. Fix the sign on a dev subset

A wrong sign yields a large **negative** normalized PRR (~-0.7), which reads as a
broken method rather than an inverted score. Settle it on ~150 examples before
the real run. The accuracy gate also fires here first — if exact match is outside
10–90%, **stop and fix the prompt**, because PRR is noise outside that band.

In [ ]:
cmd = ("python harness/run_baselines.py"
       " --config configs/stage1/eval_triviaqa_qwen.yaml"
       f" --save-dir '{DRIVE_OUT}/dev_signcheck'"
       " --samples 150 --n-boot 0"
       " --expect-model Qwen/Qwen2.5-3B-Instruct")
print(cmd)
!{cmd}

## 8. Final run — all estimators, one pass, n=1000

Baselines and SpilledEnergy share one `UEManager`, so generations and generation
settings are identical by construction (fp16 + different batch composition can
otherwise shift generations between runs).

If the sign check said a variant is inverted, set it in
`configs/stage1/estimators/stage1_baselines.yaml` (`cfg: {sign: -1}`) and commit —
do not patch it here.

In [ ]:
cmd = ("python harness/run_baselines.py"
       " --config configs/stage1/eval_triviaqa_qwen.yaml"
       f" --save-dir '{DRIVE_OUT}/final_n1000'"
       " --n-boot 1000"
       " --expect-model Qwen/Qwen2.5-3B-Instruct")
print(cmd)
!{cmd}

## 9. The reported table

`per_sample_seed1.npz` is on Drive too — the table can be rebuilt offline on CPU
with `python harness/run_baselines.py --skip-run --save-dir <dir>`.

In [ ]:
from IPython.display import Markdown, display
display(Markdown((DRIVE_OUT / 'final_n1000' / 'prr_0.5_table.md').read_text()))

import torch, transformers, datasets, sys
print('python      ', sys.version.split()[0])
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('device      ', torch.cuda.get_device_name(0))
!git rev-parse HEAD